# Legal-RAG Colab: Chunk -> Embed -> Upsert

Notebook này chỉ giữ pipeline indexing:
1. Chunk văn bản pháp luật.
2. Tạo dense + sparse embedding bằng BGE-M3.
3. In thử 1 payload mẫu.
4. Upsert lên Qdrant.

Không dùng LLM chat trong notebook này.

In [ ]:
# @title Cấu hình runtime + Qdrant + HF
import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

COLLECTION_NAME = "legal_rag_docs_5000" # @param {type:"string"}
DOC_LIMIT = 5000 # @param {type:"integer"}

if IN_KAGGLE:
    QDRANT_LOCAL_PATH = "/kaggle/working/local_qdrant_data"
elif IN_COLAB:
    QDRANT_LOCAL_PATH = "/content/local_qdrant_data"
else:
    QDRANT_LOCAL_PATH = "./local_qdrant_data"

QDRANT_LOCAL_URL = "http://localhost:6333" # @param {type:"string"}
QDRANT_MODE = "cloud" # @param ["local_path", "local_url", "cloud"]
QDRANT_URL = "https://bc0bb490-f62b-46b6-9863-5e413732dd93.us-west-1-0.aws.cloud.qdrant.io:6333" # @param {type:"string"}
QDRANT_API_KEY = "" # @param {type:"string"}
HF_API_KEY = "" # @param {type:"string"}

if QDRANT_MODE == "local_path":
    Path(QDRANT_LOCAL_PATH).mkdir(parents=True, exist_ok=True)

os.environ["QDRANT_MODE"] = QDRANT_MODE
os.environ["QDRANT_LOCAL_PATH"] = QDRANT_LOCAL_PATH
os.environ["QDRANT_LOCAL_URL"] = QDRANT_LOCAL_URL
os.environ["QDRANT_URL"] = QDRANT_URL
os.environ["QDRANT_API_KEY"] = QDRANT_API_KEY
os.environ["DOC_LIMIT"] = str(DOC_LIMIT)
os.environ["HF_TOKEN"] = HF_API_KEY

try:
    if HF_API_KEY:
        from huggingface_hub import login
        login(token=HF_API_KEY, add_to_git_credential=False)
        print("Hugging Face login: OK")
except Exception as exc:
    print(f"Hugging Face login warning: {exc}")

print(f"IN_KAGGLE={IN_KAGGLE} | IN_COLAB={IN_COLAB}")
print(f"QDRANT_MODE={QDRANT_MODE} | COLLECTION={COLLECTION_NAME}")
print(f"DOC_LIMIT={DOC_LIMIT} (0 = all)")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login: OK
IN_KAGGLE=False | IN_COLAB=True
QDRANT_MODE=cloud | COLLECTION=legal_rag_docs_5000
DOC_LIMIT=5000 (0 = all)


In [ ]:
%pip install -q -U qdrant-client datasets pandas accelerate python-dotenv ipywidgets sentence-transformers fastembed pypdf python-docx "FlagEmbedding==1.3.5" "transformers==4.48.3" huggingface-hub tokenizers

In [ ]:
# Setup Qdrant client + BGE-M3 hybrid encoder
import os
from pathlib import Path
from typing import Dict, List, Tuple

import torch
from qdrant_client import QdrantClient, models
from FlagEmbedding import BGEM3FlagModel

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DENSE_MODEL_NAME = os.getenv("LEGAL_DENSE_MODEL", "BAAI/bge-m3")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QDRANT_MODE = os.getenv("QDRANT_MODE", "cloud").lower()
QDRANT_LOCAL_PATH = os.getenv("QDRANT_LOCAL_PATH", "./local_qdrant_data")
QDRANT_LOCAL_URL = os.getenv("QDRANT_LOCAL_URL", "http://localhost:6333")
QDRANT_CLOUD_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
COLLECTION_NAME = globals().get("COLLECTION_NAME", "legal_rag_docs_5000")


def init_qdrant_client() -> QdrantClient:
    if QDRANT_MODE == "local_path":
        Path(QDRANT_LOCAL_PATH).mkdir(parents=True, exist_ok=True)
        client = QdrantClient(path=QDRANT_LOCAL_PATH)
        client.get_collections()
        print(f"Qdrant connected (local_path): {QDRANT_LOCAL_PATH}")
        return client

    if QDRANT_MODE == "local_url":
        client = QdrantClient(url=QDRANT_LOCAL_URL)
        client.get_collections()
        print(f"Qdrant connected (local_url): {QDRANT_LOCAL_URL}")
        return client

    if not QDRANT_CLOUD_URL:
        raise ValueError("QDRANT_URL đang rỗng trong cloud mode.")

    client = QdrantClient(url=QDRANT_CLOUD_URL, api_key=(QDRANT_API_KEY or None))
    client.get_collections()
    print(f"Qdrant connected (cloud): {QDRANT_CLOUD_URL}")
    return client


class LocalBGEHybridEncoder:
    def __init__(self, model_name: str = "BAAI/bge-m3", device: str = "cpu"):
        self.model = BGEM3FlagModel(model_name, use_fp16=(device == "cuda"), device=device)

    @staticmethod
    def _to_sparse_vector(weights: Dict[str, float]) -> models.SparseVector:
        if not weights:
            return models.SparseVector(indices=[], values=[])
        pairs = [(int(k), float(v)) for k, v in weights.items() if float(v) != 0.0]
        pairs.sort(key=lambda x: x[0])
        return models.SparseVector(indices=[i for i, _ in pairs], values=[v for _, v in pairs])

    def encode_hybrid(self, texts: List[str], batch_size: int = 16) -> Tuple[List[List[float]], List[models.SparseVector]]:
        if isinstance(texts, str):
            texts = [texts]

        out = self.model.encode(
            texts,
            batch_size=(128 if torch.cuda.is_available() else batch_size),
            max_length=2048,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
        )

        dense_vecs = out["dense_vecs"]
        dense_list = dense_vecs.tolist() if hasattr(dense_vecs, "tolist") else [list(v) for v in dense_vecs]
        sparse_list = [self._to_sparse_vector(w) for w in out["lexical_weights"]]
        return dense_list, sparse_list


qdrant_client = init_qdrant_client()
hybrid_encoder = LocalBGEHybridEncoder(model_name=DENSE_MODEL_NAME, device=DEVICE)

dense_dim = len(hybrid_encoder.encode_hybrid(["probe"], batch_size=1)[0][0])
print(f"Device={DEVICE} | dense_dim={dense_dim}")

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(size=dense_dim, distance=models.Distance.COSINE, on_disk=True)
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(index=models.SparseIndexParams(on_disk=False))
        },
    )
    print(f"Created collection: {COLLECTION_NAME}")
else:
    print(f"Collection exists: {COLLECTION_NAME}")

Qdrant connected (cloud): https://bc0bb490-f62b-46b6-9863-5e413732dd93.us-west-1-0.aws.cloud.qdrant.io:6333


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

colbert_linear.pt:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Device=cuda | dense_dim=1024
Created collection: legal_rag_docs_5000


In [ ]:
# Load dataset + lọc metadata/content theo DOC_LIMIT
import os
import re
from typing import Any, List
import pandas as pd
from datasets import load_dataset
from IPython.display import display

def split_sector_list(value: Any) -> List[str]:
    if pd.isna(value) or not value:
        return []
    if isinstance(value, list):
        raw_text = ", ".join(str(x) for x in value if str(x).strip())
    else:
        raw_text = str(value).strip()
    if not raw_text or raw_text.lower() == "nan":
        return []
    parts = re.split(r"\s*,\s*", raw_text)
    return sorted(set(part.strip() for part in parts if part and part.strip()))

def parse_date(date_str: Any) -> str:
    """Đổi ngày tháng về chuẩn YYYY-MM-DD"""
    if pd.isna(date_str) or not str(date_str).strip():
        return ""
    date_str = str(date_str).strip()
    try:
        # Trường hợp '2026-03-16 00:00:00'
        if " " in date_str and "-" in date_str:
            return date_str.split(" ")[0]
        # Trường hợp '29/01/2026'
        elif "/" in date_str:
            parts = date_str.split("/")
            if len(parts) == 3:
                return f"{parts[2]}-{parts[1]}-{parts[0]}"
    except:
        pass
    return date_str

def parse_signer_name(signer_str: Any) -> str:
    """Tách lấy Tên người ký"""
    if pd.isna(signer_str) or not str(signer_str).strip():
        return ""
    return str(signer_str).split(":")[0].strip()

def parse_signer_id(signer_str: Any) -> Any:
    """Tách lấy ID người ký"""
    if pd.isna(signer_str) or not str(signer_str).strip():
        return None
    parts = str(signer_str).split(":")
    if len(parts) > 1 and parts[1].strip().isdigit():
        return int(parts[1].strip())
    return None

print("Loading dataset...")
ds_metadata_all = load_dataset("nhn309261/vietnamese-legal-documents", "metadata", split="data")
ds_content_all = load_dataset("nhn309261/vietnamese-legal-documents", "content", split="data")

# 1. Đưa vào Pandas để biến đổi dữ liệu siêu tốc
df_all = ds_metadata_all.to_pandas().copy()
df_all["id"] = df_all["id"].astype(str)

# --- CHUẨN HÓA METADATA CHO "GOLDEN CHUNK PAYLOAD" ---
df_all["legal_sectors_list"] = df_all["legal_sectors"].apply(split_sector_list)

# Thời gian
df_all["promulgation_date"] = df_all["issuance_date"].apply(parse_date)
df_all["effective_date"] = "" # Data gốc không có, gán rỗng để backend RAG không bị lỗi

# Người ký (Xử lý cột signers)
df_all["signer_name"] = df_all["signers"].apply(parse_signer_name)
df_all["signer_id"] = df_all["signers"].apply(parse_signer_id)
# -----------------------------------------------------

valid_legal_types = ["Luật", "Nghị định", "Quyết định", "Thông tư", "Hiến pháp", "Bộ luật", "Pháp lệnh", "Nghị quyết"]
df_all = df_all[df_all["legal_type"].isin(valid_legal_types)].copy()

content_ids = {str(row["id"]) for row in ds_content_all}
valid_ids_all = set(df_all["id"]).intersection(content_ids)

df_valid = df_all[df_all["id"].isin(valid_ids_all)].copy()

DOC_LIMIT = int(os.getenv("DOC_LIMIT", str(globals().get("DOC_LIMIT", 0))) or 0)
if DOC_LIMIT > 0:
    # TÍNH NĂNG MỚI: Sắp xếp lấy văn bản mới nhất trước khi cắt LIMIT
    df_valid = df_valid.sort_values(by="promulgation_date", ascending=False)
    df_valid = df_valid.head(DOC_LIMIT)

selected_ids = df_valid["id"].tolist()
selected_id_set = set(selected_ids)

# 2. Tối ưu hiệu năng: Dùng trực tiếp df_valid làm Dictionary (nhanh gấp 10 lần HF filter)
# Không dùng set_index nữa, giữ nguyên cột id trong dictionary
metadata_by_id = df_valid.to_dict(orient="index")
# Sửa lại cách lấy dict một chút:
metadata_by_id = {str(row['id']): row for row in df_valid.to_dict(orient="records")}

# Đối với content thì bắt buộc phải dùng HF filter do nội dung dài
ds_content_selected = ds_content_all.filter(lambda row: str(row["id"]) in selected_id_set)
content_by_id = {str(row["id"]): row for row in ds_content_selected}

print(f"Total valid docs: {len(valid_ids_all)}")
print(f"Selected docs for indexing: {len(ds_content_selected)}")

# Print stats
sector_stats = df_valid.explode("legal_sectors_list")["legal_sectors_list"].dropna().value_counts()
display(sector_stats.head(15).to_frame("count"))

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

metadata/data-00000-of-00001.parquet:   0%|          | 0.00/81.8M [00:00<?, ?B/s]

Generating data split: 0 examples [00:00, ? examples/s]

content/data-00000-of-00011.parquet:   0%|          | 0.00/424M [00:00<?, ?B/s]

...

content/data-00010-of-00011.parquet:   0%|          | 0.00/124M [00:00<?, ?B/s]

Generating data split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/518255 [00:00<?, ? examples/s]

Total valid docs: 314416
Selected docs for indexing: 5000


,count
legal_sectors_list,
Bộ máy hành chính,2533
Tài chính nhà nước,863
Văn hóa - Xã hội,620
Tài nguyên - Môi trường,439
Thể thao - Y tế,321
Xây dựng - Đô thị,286
Bất động sản,280
Thương mại,214
Lao động - Tiền lương,209


# Tiếp tục từ đây: Chunk lại, lọc metadata


In [75]:
import re
import uuid
from typing import Any, Dict, List

def compact_whitespace(text: str) -> str:
    return re.sub(r"[ \t]+", " ", str(text or "")).strip()

class AdvancedLegalChunker:
    def __init__(self):
        # 1. NHẬN DIỆN PHỤ LỤC (Mở rộng)
        self.appendix_pattern = re.compile(
            r"(?im)^\s*("
            r"(?:Mẫu\s+số|Mẫu|Biểu\s+mẫu)[\s\d\w\.\-\:]*|" # Bắt mọi cụm bắt đầu bằng Mẫu số, Mẫu...
            r"PHỤ LỤC|PHU LUC|"
            r"DANH MỤC|DANH MUC|"
            r"BẢNG BIỂU|BANG BIEU|"
            r"PHƯƠNG ÁN|PHUONG AN|"
            r"QUY ĐỊNH|QUY DINH|"
            r"QUY CHẾ|QUY CHE"
            r")\b.*$"
        )

        self.chapter_pattern = re.compile(r"(?im)^\s*(Chương\s+[IVXLCDM0-9]+)\s*(.*)$")
        self.article_pattern = re.compile(r"(?im)^\s*(Điều\s+\d+[A-Za-z0-9\/\-]*)\s*[\.\:\-]?\s*(.*)$")
        self.clause_pattern = re.compile(r"(?im)^\s*(Khoản\s+\d+[\.\:\-]?)\s*(.*)$|^\s*(\d+[\.\)])\s*(.*)$")

        self.legal_basis_line_pattern = re.compile(r"(?im)^\s*căn cứ\b.*$")
        self.legal_ref_pattern = re.compile(r"(?i)\b(Hiến pháp|Bộ luật|Luật|Nghị quyết|Pháp lệnh|Nghị định|Thông tư)\b([^.;\n]*)")
        self.effective_date_pattern = re.compile(r"(?i)có\s+hiệu\s+lực\s+(?:từ|kể\s+từ)\s+ngày\s+(\d{1,2})[/\- ](\d{1,2})[/\- ](\d{4})")
        self.effective_from_sign_pattern = re.compile(
            r"(?i)("
            r"có\s+hiệu\s+lực\s+(?:thi\s+hành\s+)?(?:kể\s+)?từ\s+ngày\s+ký|" # Loại 1: Hiệu lực từ ngày ký
            r"chịu\s+trách\s+nhiệm\s+thi\s+hành\s+(?:quyết\s+định|nghị\s+định|thông\s+tư|văn\s+bản)\s+này" # Loại 2: Chịu trách nhiệm thi hành
            r")"
        )

        # 2. NHẬN DIỆN ĐIỀU KHOẢN KẾT THÚC VÀ FOOTER
        self.final_article_trigger = re.compile(
            r"(?i)("
            r"có\s+hiệu\s+lực\s+(?:thi\s+hành\s+)?(?:kể\s+)?từ\s+ngày|"
            r"chịu\s+trách\s+nhiệm\s+thi\s+hành|"
            r"tổ\s+chức\s+thực\s+hiện"
            r")"
        )
        self.footer_pattern = re.compile(
            # Bắt chính xác cụm "Nơi nhận:" hoặc "Kính gửi:" ở đầu dòng
            r"(?i)^\s*(nơi\s+nhận|kính\s+gửi)[\:\.]?|"

            # Bắt các tiền tố chữ ký (Thay mặt, Ký thay, Thừa lệnh...)
            r"^\s*(TM\.|KT\.|Q\.|TL\.|TUQ\.)?\s*"

            # Bắt ĐẦY ĐỦ các chức danh lãnh đạo phổ biến nhất của bộ máy Nhà nước
            r"("
            r"CHÍNH\s+PHỦ|UBND|ỦY\s+BAN\s+NHÂN\s+DÂN|"
            r"BỘ\s+TRƯỞNG|CHỦ\s+TỊCH|THỨ\s+TRƯỞNG|GIÁM\s+ĐỐC|TỔNG\s+GIÁM\s+ĐỐC|"
            r"CỤC\s+TRƯỞNG|TỔNG\s+CỤC\s+TRƯỞNG|CHÁNH\s+VĂN\s+PHÒNG|"
            r"CHÁNH\s+ÁN|VIỆN\s+TRƯỞNG|TỔNG\s+KIỂM\s+TOÁN|CHỦ\s+NHIỆM|TỔNG\s+BÍ\s+THƯ"
            r")\b"
        )


    @staticmethod
    def _slugify(value: str) -> str:
        return re.sub(r"[^a-z0-9]+", "-", (value or "").lower()).strip("-") or "unknown"

    @staticmethod
    def _canonical_doc_type(raw: str) -> str:
        text = (raw or "").lower()
        mapping = {
            "hiến pháp": "constitution", "hien phap": "constitution",
            "bộ luật": "code", "bo luat": "code",
            "luật": "law", "luat": "law",
            "pháp lệnh": "ordinance", "phap lenh": "ordinance",
            "nghị quyết": "resolution", "nghi quyet": "resolution",
            "nghị định": "decree", "nghi dinh": "decree",
            "thông tư": "circular", "thong tu": "circular"
        }
        for k, v in mapping.items():
            if k in text: return v
        return "other"

    @staticmethod
    def _extract_year(text: str) -> str:
        m = re.search(r"\b(19|20)\d{2}\b", text or "")
        return m.group(0) if m else ""

    @staticmethod
    def _extract_doc_number(text: str) -> str:
        patterns = [r"(?i)(?:số\s*)?(\d+\/\d+(?:\/[A-Z0-9Đ\-]+)?)", r"(?i)(\d+\/[A-Z0-9Đ\-]+)"]
        for p in patterns:
            m = re.search(p, text or "")
            if m: return compact_whitespace(m.group(1))
        return ""

    def _build_parent_law_id(self, doc_type: str, doc_number: str, year: str, doc_title: str) -> str:
        basis = doc_number or doc_title or "unknown"
        return f"parent::{doc_type}::{self._slugify(basis)}::{year or 'unknown'}"

    def _parse_legal_basis_line(self, raw_line: str):
        refs = []
        line = compact_whitespace(raw_line)
        for m in self.legal_ref_pattern.finditer(line):
            raw_type = compact_whitespace(m.group(1))
            tail = compact_whitespace(m.group(2))
            full_ref = compact_whitespace(f"{raw_type} {tail}")
            doc_type = self._canonical_doc_type(raw_type)
            year = self._extract_year(full_ref)
            doc_number = self._extract_doc_number(full_ref)
            refs.append({
                "basis_line": line, "doc_type": doc_type, "doc_number": doc_number,
                "doc_year": year, "doc_title": full_ref,
                "parent_law_id": self._build_parent_law_id(doc_type, doc_number, year, full_ref),
            })
        return refs

    def _extract_legal_basis_metadata(self, content: str) -> List[dict]:
        preamble = "\n".join((content or "").splitlines()[:80])
        all_refs = []
        for line in preamble.splitlines():
            if self.legal_basis_line_pattern.match(line):
                all_refs.extend(self._parse_legal_basis_line(line))
        dedup = []
        seen = set()
        for r in all_refs:
            key = (r.get("parent_law_id"), r.get("doc_title"))
            if key in seen: continue
            seen.add(key)
            dedup.append(r)
        return dedup

    def _extract_effective_date(self, content: str, promulgation_date: str) -> str:
        lines = content.splitlines()
        appendix_start_idx = len(lines)
        for i, line in enumerate(lines):
            if self.appendix_pattern.match(line):
                appendix_start_idx = i
                break
        main_body_lines = lines[:appendix_start_idx]
        search_lines = main_body_lines[:200]
        if len(main_body_lines) > 200:
            search_lines += main_body_lines[-200:]
        search_zone = "\n".join(search_lines)
        m = self.effective_date_pattern.search(search_zone)
        if m:
            day, month, year = m.groups()
            return f"{year}-{month.zfill(2)}-{day.zfill(2)}"
        if self.effective_from_sign_pattern.search(search_zone):
            return promulgation_date
        return ""

    @staticmethod
    def _parse_signer(signer_raw: str) -> tuple:
        if not signer_raw or ":" not in signer_raw:
            return signer_raw, None
        parts = signer_raw.split(":")
        name = parts[0].strip()
        try:
            return name, int(parts[1].strip())
        except:
            return name, None

    def process_document(self, content: str, metadata: Dict[str, Any]) -> List[Dict[str, Any]]:
        content = str(content or "").replace("\r\n", "\n").strip()
        doc_id = str(metadata.get("id", uuid.uuid4()))
        doc_number = metadata.get("document_number", "N/A")
        signer_name, signer_id = self._parse_signer(metadata.get("signers", ""))
        promulgation_date = metadata.get("promulgation_date", "")
        eff_date = self._extract_effective_date(content, promulgation_date)
        final_effective_date = eff_date or metadata.get("effective_date", "") or promulgation_date
        basis_refs = self._extract_legal_basis_metadata(content)

        chunks: List[Dict[str, Any]] = []

        # --- State Tracking ---
        current_chapter = None
        current_article_ref = None
        current_article_title = ""
        current_clauses_data = []
        article_preamble = []
        current_appendix_buffer = []

        is_in_appendix = False
        current_appendix_name = ""
        current_article_idx = 0
        current_appendix_idx = 0
        global_chunk_idx = 0
        found_final_article = False

        def flush_buffer():
            nonlocal global_chunk_idx, current_clauses_data, article_preamble, current_appendix_buffer
            if not current_clauses_data and not article_preamble and not current_appendix_buffer:
                return

            parts = []

            if is_in_appendix:
                parts.extend(current_appendix_buffer)
                cl_refs = []
            else:
                # [GIẢI QUYẾT YÊU CẦU 1]: Luôn đính kèm lời dẫn (preamble) vào mọi Point của Điều
                if article_preamble:
                    parts.append("\n".join(article_preamble))

                cl_refs = [c["ref"] for c in current_clauses_data]
                if current_clauses_data:
                    parts.extend([c["text"] for c in current_clauses_data])

            text_content = "\n".join(parts).strip()
            if not text_content: return

            global_chunk_idx += 1

            if is_in_appendix:
                breadcrumb = current_appendix_name if current_appendix_name else "Phụ lục"
                chunk_id_val = f"{doc_id}::appendix::{current_appendix_idx}::c{global_chunk_idx}"
                ref_citation = f"{doc_number} | {breadcrumb}"
                clause_label = "PHỤ LỤC"
                art_ref = None
                cl_ref_list = []
                current_appendix_buffer.clear()
            else:
                art_full = f"{current_article_ref}. {current_article_title}".strip(". ") if current_article_ref else None

                if cl_refs:
                    cl_display = f"{cl_refs[0]} - {cl_refs[-1]}" if len(cl_refs) > 1 else cl_refs[0]
                    clause_label = cl_display.upper()
                    cl_ref_list = cl_refs
                else:
                    clause_label = current_article_ref.upper() if current_article_ref else "CHUNG"
                    cl_ref_list = []

                bc_parts = [current_chapter, art_full, (cl_display if cl_refs else None)]
                breadcrumb = " > ".join([x for x in bc_parts if x])
                chunk_id_val = f"{doc_id}::article::{current_article_idx}::c{global_chunk_idx}"
                ref_citation = f"{doc_number} | {breadcrumb.replace(' > ', ' | ')}" if breadcrumb else doc_number
                art_ref = art_full

            chunk_text = (
                f"[VĂN BẢN] {doc_number}\n"
                f"[VỊ TRÍ] {breadcrumb}\n"
                f"[NỘI DUNG {clause_label}]\n"
                f"{text_content}"
            )

            payload = {
                "document_id": doc_id,
                "chunk_index": global_chunk_idx,
                "document_number": doc_number,
                "title": metadata.get("title", "N/A"),
                "legal_type": metadata.get("legal_type", "N/A"),
                "legal_sectors": metadata.get("legal_sectors_list", []),
                "issuing_authority": metadata.get("issuing_authority", "N/A"),
                "signer_name": signer_name,
                "signer_id": signer_id,
                "url": metadata.get("url", ""),
                "promulgation_date": promulgation_date,
                "effective_date": final_effective_date,
                "is_active": True,
                "chapter_ref": current_chapter,
                "article_ref": art_ref,
                "clause_ref": cl_ref_list,
                "is_appendix": is_in_appendix,
                "reference_citation": ref_citation,
                "chunk_text": chunk_text,
                "legal_basis_refs": basis_refs
            }

            chunks.append({
                "chunk_id": chunk_id_val,
                "text_to_embed": chunk_text,
                "metadata": payload
            })

        # --- XỬ LÝ TỪNG DÒNG TEXT ---
        lines = content.splitlines()
        for line in lines:
            line = line.strip()
            if not line: continue

            # [0] KIỂM TRA CÂU HIỆU LỰC NGAY LẬP TỨC TRƯỚC KHI BỊ LỆNH CONTINUE BỎ QUA
            if not is_in_appendix and not found_final_article:
                if self.final_article_trigger.search(line):
                    found_final_article = True

            # [1] KIỂM TRA PHỤ LỤC
            m_app = self.appendix_pattern.match(line)
            if m_app:
                flush_buffer()
                current_clauses_data = []
                article_preamble = []
                is_in_appendix = True
                current_appendix_idx += 1
                current_appendix_name = compact_whitespace(line)
                current_chapter = None
                current_article_ref = None
                current_article_title = ""
                continue

            # [2] KIỂM TRA FOOTER
            if not is_in_appendix and found_final_article:
                if self.footer_pattern.match(line) or (line.isupper() and len(line) > 5):
                    flush_buffer()
                    current_clauses_data = []
                    article_preamble = []
                    is_in_appendix = True
                    current_appendix_idx += 1
                    current_appendix_name = "Nơi nhận / Footer" if re.search(r"(?i)^(nơi\s+nhận|kính\s+gửi)", line) else "Chữ ký / Đóng dấu"
                    current_chapter = None
                    current_article_ref = None
                    current_article_title = ""
                    current_appendix_buffer.append(line)
                    continue

            # [3] BẪY BẢNG BIỂU (MARKDOWN TABLE) SAU ĐIỀU CUỐI CÙNG
            if not is_in_appendix and found_final_article:
                if line.startswith("|"):
                    flush_buffer()
                    current_clauses_data = []
                    article_preamble = []
                    is_in_appendix = True
                    current_appendix_idx += 1
                    current_appendix_name = "Bảng biểu / Phụ lục đính kèm"
                    current_chapter = None
                    current_article_ref = None
                    current_article_title = ""
                    current_appendix_buffer.append(line)
                    continue

            # Xử lý Phụ lục
            if is_in_appendix:
                MAX_APP_LEN = 2000
                current_buffer_len = sum(len(s) for s in current_appendix_buffer)
                line_len = len(line)

                if current_buffer_len > 0 and (current_buffer_len + line_len) > MAX_APP_LEN:
                    flush_buffer()
                    current_buffer_len = 0

                if line_len > MAX_APP_LEN:
                    for i in range(0, line_len, MAX_APP_LEN):
                        sub_line = line[i : i + MAX_APP_LEN]
                        current_appendix_buffer.append(sub_line)
                        flush_buffer()
                else:
                    current_appendix_buffer.append(line)
                    if current_buffer_len + line_len > MAX_APP_LEN:
                        flush_buffer()
                continue

            # Phân tích Chương
            m_ch = self.chapter_pattern.match(line)
            if m_ch:
                flush_buffer()
                current_clauses_data = []
                article_preamble = []
                current_chapter = compact_whitespace(f"{m_ch.group(1)}. {m_ch.group(2)}")
                continue

            # Phân tích Điều
            m_ar = self.article_pattern.match(line)
            if m_ar:
                flush_buffer()
                current_clauses_data = []
                article_preamble = []
                current_article_idx += 1
                current_article_ref = m_ar.group(1).strip()
                article_remainder = m_ar.group(2).strip()
                current_article_title = article_remainder[:300] + "..." if len(article_remainder) > 300 else article_remainder

                if article_remainder:
                    article_preamble.append(article_remainder)
                continue

            # Phân tích Khoản
            m_cl = self.clause_pattern.match(line)
            if m_cl and current_article_ref:
                # [GIẢI QUYẾT YÊU CẦU 2]: Xả thông minh TRƯỚC khi nạp Khoản mới
                current_len = sum(len(c["text"]) for c in current_clauses_data) + sum(len(p) for p in article_preamble)
                num_clauses = len(current_clauses_data)

                should_flush = False

                if num_clauses >= 3:
                    should_flush = True # Đã đủ 3 khoản -> Xả
                elif num_clauses == 2 and current_len > 1500:
                    should_flush = True # 2 khoản nhưng khá nặng -> Xả sớm
                elif num_clauses >= 1 and current_len > 2500:
                    should_flush = True # Dù mới có 1-2 khoản nhưng đã quá to (>2500) -> Xả để nhường chỗ cho Khoản mới

                if should_flush:
                    last_clause = current_clauses_data[-1]
                    flush_buffer()

                    if num_clauses > 1:
                        current_clauses_data = [last_clause] # Chồng lấn 1 khoản
                    else:
                        current_clauses_data = []

                cl_ref = compact_whitespace(m_cl.group(1) or m_cl.group(3))
                cl_text = compact_whitespace(m_cl.group(2) or m_cl.group(4))

                current_clauses_data.append({
                    "ref": cl_ref,
                    "text": f"{cl_ref} {cl_text}".strip() if cl_text else cl_ref
                })
                continue

            # ==========================================
            # TEXT BÌNH THƯỜNG & CƠ CHẾ TÁCH HỒI TỐ
            # ==========================================
            if len(line) > 3000 and not is_in_appendix:
                flush_buffer()
                is_in_appendix = True
                current_appendix_name = "Nội dung bảng biểu siêu dài"
                current_appendix_buffer.append(line)
                continue

            if current_clauses_data:
                current_len = sum(len(c["text"]) for c in current_clauses_data) + sum(len(p) for p in article_preamble)
                projected_len = current_len + len(line) + 1

                if projected_len > 4000:
                    if len(current_clauses_data) > 1:
                        active_monster = current_clauses_data.pop()
                        flush_buffer()
                        active_monster["text"] += "\n" + line
                        current_clauses_data = [active_monster]
                    else:
                        flush_buffer()
                        last_ref = current_clauses_data[0]["ref"] if current_clauses_data else "Khoản"
                        current_clauses_data = [{"ref": last_ref, "text": f"{last_ref} (tiếp theo)\n{line}"}]
                else:
                    current_clauses_data[-1]["text"] += "\n" + line

            # XỬ LÝ LỜI DẪN (PREAMBLE)
            else:
                projected_preamble_len = sum(len(p) for p in article_preamble) + len(line)
                # KỂ KIỂM TRA TRƯỚC KHI APPEND ĐỂ TRÁNH PHÌNH TO CHUNK
                if projected_preamble_len > 4000:
                    flush_buffer()
                    article_preamble = ["[Nội dung lời dẫn quá dài, phần trước đã được chuyển sang Chunk trước...]", line]
                else:
                    article_preamble.append(line)

            # --- Gắn cờ nếu phát hiện đây là Điều khoản hiệu lực ---


        flush_buffer()
        return chunks

chunker = AdvancedLegalChunker()
print("Advanced Chunker with Footer/Appendix Trap is ready.")

Advanced Chunker with Footer/Appendix Trap is ready.


In [76]:
import time

def estimate_total_chunks(dataset, metadata_dict):
    print("🚀 Đang quét toàn bộ dữ liệu để tính toán quy mô... (Chỉ chạy CPU, không tốn GPU)")
    t_start = time.perf_counter()

    total_docs = len(dataset)
    total_chunks = 0
    doc_count = 0

    # --- Biến lưu trữ thông tin của Chunk dài nhất ---
    max_chunk_len = 0
    max_chunk_doc = ""
    max_chunk_position = ""
    max_chunk_text = ""

    # Chỉ chạy logic cắt text, không nhúng
    for row in dataset:
        doc_id = str(row["id"])
        meta = metadata_dict.get(doc_id)
        if not meta:
            continue

        text_content = str(row.get("text", row.get("content", "")))

        # Gọi hàm chunker
        doc_chunks = chunker.process_document(text_content, meta)
        total_chunks += len(doc_chunks)
        doc_count += 1

        # --- TÌM CHUNK DÀI NHẤT TRONG TÀI LIỆU HIỆN TẠI ---
        for chunk in doc_chunks:
            # Lấy nội dung text của chunk (dựa theo cấu trúc dict bạn đang return)
            current_chunk_text = chunk.get("text_to_embed", "")
            current_len = len(current_chunk_text)

            # Nếu phát hiện chunk dài hơn kỷ lục hiện tại -> Cập nhật
            if current_len > max_chunk_len:
                max_chunk_len = current_len
                chunk_meta = chunk.get("metadata", {})

                max_chunk_doc = chunk_meta.get("document_number", "N/A")
                max_chunk_position = chunk_meta.get("reference_citation", "N/A")
                max_chunk_text = current_chunk_text

        if doc_count % 500 == 0:
            print(f"--- Đã quét {doc_count}/{total_docs} văn bản... ---")

    duration = time.perf_counter() - t_start

    print("\n" + "="*50)
    print("📊 BÁO CÁO DỰ KIẾN (ESTIMATION REPORT)")
    print("="*50)
    print(f"🔹 Tổng số văn bản: {doc_count:,}")
    print(f"🔹 Tổng số chunk dự kiến: {total_chunks:,}")
    print(f"🔹 Tỷ lệ trung bình: {total_chunks/doc_count:.2f} chunks/văn bản")
    print(f"⏱️ Thời gian quét: {duration:.2f} giây")

    # Ước tính dung lượng Qdrant (Dense 1024 + Sparse + Metadata ~ 10KB mỗi point)
    est_size_mb = (total_chunks * 10) / 1024
    print(f"💾 Ước tính dung lượng DB: ~{est_size_mb:.2f} MB")

    print("\n" + "="*50)
    print("⚠️ THÔNG TIN CHUNK DÀI NHẤT (MAX LENGTH CHUNK)")
    print("="*50)
    print(f"🔸 Số ký tự (Max Len): {max_chunk_len:,} characters")
    print(f"🔸 Số hiệu văn bản: {max_chunk_doc}")
    print(f"🔸 Vị trí (Breadcrumb): {max_chunk_position}")
    print("-" * 50)
    print("🔸 NỘI DUNG CHUNK:\n")
    print(max_chunk_text)
    print("="*50)

# Chạy thử
estimate_total_chunks(ds_content_selected, metadata_by_id)

🚀 Đang quét toàn bộ dữ liệu để tính toán quy mô... (Chỉ chạy CPU, không tốn GPU)
--- Đã quét 500/5000 văn bản... ---
--- Đã quét 1000/5000 văn bản... ---
--- Đã quét 1500/5000 văn bản... ---
--- Đã quét 2000/5000 văn bản... ---
--- Đã quét 2500/5000 văn bản... ---
--- Đã quét 3000/5000 văn bản... ---
--- Đã quét 3500/5000 văn bản... ---
--- Đã quét 4000/5000 văn bản... ---
--- Đã quét 4500/5000 văn bản... ---
--- Đã quét 5000/5000 văn bản... ---

📊 BÁO CÁO DỰ KIẾN (ESTIMATION REPORT)
🔹 Tổng số văn bản: 5,000
🔹 Tổng số chunk dự kiến: 110,597
🔹 Tỷ lệ trung bình: 22.12 chunks/văn bản
⏱️ Thời gian quét: 11.72 giây
💾 Ước tính dung lượng DB: ~1080.05 MB

⚠️ THÔNG TIN CHUNK DÀI NHẤT (MAX LENGTH CHUNK)
🔸 Số ký tự (Max Len): 4,680 characters
🔸 Số hiệu văn bản: 48/2025/NQ-HĐND
🔸 Vị trí (Breadcrumb): 48/2025/NQ-HĐND | Điều 1. Sửa đổi, bổ sung và bãi bỏ một số điều của Nghị quyết số 177/2021/NQ-HĐND ngày 10 tháng 12 năm 2021 của Hội đồng nhân dân tỉnh về nguyên tắc, tiêu chí và định mức phân bổ dự

In [ ]:
# @title Cell Test Parser: Kiểm tra văn bản theo SỐ HIỆU (document_number)
import json

def run_test_by_doc_number(target_number="142/QĐ-QLD"):
    try:
        # 1. Kiểm tra dữ liệu đã load chưa
        if 'content_by_id' not in globals() or 'metadata_by_id' not in globals():
            print("❌ Lỗi: Chưa load dữ liệu từ Hugging Face. Hãy chạy cell load dataset trước!")
            return

        print(f"🔍 Đang tìm kiếm văn bản có số hiệu: {target_number}...")

        # Tìm ID tương ứng với document_number
        target_doc_id = None
        for doc_id, meta in metadata_by_id.items():
            if meta.get('document_number') == target_number:
                target_doc_id = doc_id
                break

        if not target_doc_id:
            print(f"⚠️ Không tìm thấy văn bản có số hiệu {target_number} trong danh sách đã load.")
            print("Gợi ý: Kiểm tra lại dấu cách hoặc chính tả (VD: 142/QĐ-QLD thay vì 142/QD-QLD)")
            return

        target_meta = metadata_by_id[target_doc_id]
        content_dict = content_by_id[target_doc_id]
        target_content = content_dict.get('text', content_dict.get('content', ''))

        print(f"✅ Đã tìm thấy văn bản ID: {target_doc_id}")
        print(f"📊 Tiêu đề: {target_meta.get('title')}")
        print("-" * 80)

        # 2. Chạy Chunker
        test_chunks = chunker.process_document(target_content, target_meta)
        print(f"🚀 Tổng số chunk cắt được: {len(test_chunks)}")

        if len(test_chunks) == 0:
            print("⚠️ Văn bản này không cắt được chunk nào.")
            return

        # 3. In toàn bộ Chunk
        print(f"\n🔍 SOI CHI TIẾT TOÀN BỘ {len(test_chunks)} CHUNK:")

        for i, c in enumerate(test_chunks):
            print("=" * 80)
            print(f"📦 CHUNK INDEX: {c['metadata']['chunk_index']}")
            print(f"🔗 TRÍCH DẪN: {c['metadata']['reference_citation']}")
            print(f"📎 KHOẢN (clause_ref): {c['metadata']['clause_ref']}")
            print(f"📎 CỜ PHỤ LỤC (is_appendix): {c['metadata']['is_appendix']}")
            print("-" * 80)
            print(c['text_to_embed'])
            print("\n")

    except Exception as e:
        print(f"❌ Lỗi: {e}")

# --- THỰC HIỆN TEST ---
# Thử nghiệm với văn bản bạn vừa yêu cầu
run_test_by_doc_number("48/2026/NĐ-CP")

In [ ]:
# Chunk -> Embed -> In 1 payload mẫu -> Upsert Qdrant
import gc
import json
import time
import uuid
import os
import sys
from contextlib import contextmanager

from qdrant_client import models

# --- BỘ "BỊT MIỆNG" TQDM/LOGS ---
@contextmanager
def suppress_stdout():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

# ==========================================
# CẤU HÌNH CHẾ ĐỘ HOẠT ĐỘNG
# ==========================================
PROCEED_UPSERT = True # TẮT (False) để in 1 payload mẫu | BẬT (True) để chạy Full và lưu DB
DOC_BATCH_SIZE = 200
EMBED_BATCH_SIZE = 128
UPSERT_BATCH_SIZE = 512

if not qdrant_client.collection_exists(COLLECTION_NAME):
    raise ValueError(f"Collection {COLLECTION_NAME} chưa tồn tại.")

all_doc_rows = list(ds_content_selected)
total_docs = len(all_doc_rows)

PIPELINE_STATS = {
    "documents": total_docs,
    "chunks": 0,
    "chunk_seconds": 0.0,
    "embed_seconds": 0.0,
    "point_build_seconds": 0.0,
    "upsert_seconds": 0.0,
}

processed_docs = 0

# ==========================================
# VÒNG LẶP CHÍNH
# ==========================================
for start in range(0, total_docs, DOC_BATCH_SIZE):
    # Tối ưu: Nếu chỉ xem mẫu (Preview), ta ép lấy đúng 1 văn bản để nhúng siêu tốc
    current_batch_size = DOC_BATCH_SIZE if PROCEED_UPSERT else 1
    end = min(start + current_batch_size, total_docs)

    if PROCEED_UPSERT:
        print(f"\n🚀 Đang xử lý tài liệu {start + 1} đến {end} / {total_docs}...")
    else:
        print(f"\n🔍 Đang trích xuất văn bản đầu tiên để làm Preview Mẫu...")

    doc_batch = all_doc_rows[start : end]
    if not doc_batch:
        continue

    # 1) Chunking
    t0 = time.perf_counter()
    batch_chunks = []
    for row in doc_batch:
        doc_id = str(row["id"])
        meta = metadata_by_id.get(doc_id)
        if not meta:
            continue
        text_content = str(row.get("text", row.get("content", "")))
        batch_chunks.extend(chunker.process_document(text_content, meta))

    PIPELINE_STATS["chunk_seconds"] += time.perf_counter() - t0

    if not batch_chunks:
        processed_docs += len(doc_batch)
        continue

    # 2) Embedding
    texts = [c["text_to_embed"] for c in batch_chunks]
    t0 = time.perf_counter()

    with suppress_stdout():
        dense_vecs, sparse_vecs = hybrid_encoder.encode_hybrid(texts, batch_size=EMBED_BATCH_SIZE)

    PIPELINE_STATS["embed_seconds"] += time.perf_counter() - t0

    # 3) Build Qdrant Points
    t0 = time.perf_counter()
    points = []

    for i, chunk in enumerate(batch_chunks):
        payload = chunk["metadata"]
        qdrant_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["chunk_id"]))

        # ----------------------------------------------------
        # MODE 2: IN 1 PAYLOAD MẪU DUY NHẤT
        # ----------------------------------------------------
        if not PROCEED_UPSERT:
            if i == 2: # Chỉ khóa mục tiêu vào đúng Chunk đầu tiên
                sparse_v = sparse_vecs[i]
                sparse_len = len(sparse_v.indices) if hasattr(sparse_v, 'indices') else len(sparse_v)
                print("\n" + "="*80)
                print("🔍 [MODE 2] PREVIEW: 1 PAYLOAD MẪU DUY NHẤT")
                print("="*80)
                print(f"📦 CHUNK INDEX: {payload.get('chunk_index')} | ID: {qdrant_id}")
                print(f"📐 Vector Dense: {len(dense_vecs[i])} dims | Vector Sparse: {sparse_len} points")
                print("-" * 80)
                print("📄 METADATA JSON SẼ LƯU VÀO QDRANT:")
                print(json.dumps(payload, ensure_ascii=False, indent=2))
                print("-" * 80)
                print("📝 TEXT ĐỂ ĐƯA VÀO MODEL NHÚNG:")
                print(chunk["text_to_embed"])
                print("="*80)
            continue # Nếu đang Preview, In xong thì bỏ qua, không add vào points

        # ----------------------------------------------------
        # MODE 1: LÀM FULL
        # ----------------------------------------------------
        points.append(
            models.PointStruct(
                id=qdrant_id,
                vector={"dense": dense_vecs[i], "sparse": sparse_vecs[i]},
                payload=payload,
            )
        )

    PIPELINE_STATS["point_build_seconds"] += time.perf_counter() - t0

    # 4) Upsert (Chỉ thực thi khi PROCEED_UPSERT = True)
    if PROCEED_UPSERT:
        t0 = time.perf_counter()
        for p_start in range(0, len(points), UPSERT_BATCH_SIZE):
            p_batch = points[p_start : p_start + UPSERT_BATCH_SIZE]
            qdrant_client.upsert(collection_name=COLLECTION_NAME, points=p_batch, wait=True)
        PIPELINE_STATS["upsert_seconds"] += time.perf_counter() - t0

        PIPELINE_STATS["chunks"] += len(batch_chunks)
        processed_docs += len(doc_batch)
        print(f"✅ Xong batch! Lũy kế: {PIPELINE_STATS['chunks']:,} chunks.")

    # Dọn dẹp RAM
    del batch_chunks, texts, dense_vecs, sparse_vecs, points
    gc.collect()

    # Nếu đang ở chế độ Preview, thoát ngay lập tức sau 1 văn bản đầu tiên
    if not PROCEED_UPSERT:
        print("\n⚠️ Đã in xong 1 bản Preview. Dừng tiến trình để bạn kiểm tra!")
        break

print("\n" + "="*40)
print("🏁 TIẾN TRÌNH HOÀN TẤT")
print(f"Chế độ Upsert (Làm full): {'BẬT (Đã lưu DB)' if PROCEED_UPSERT else 'TẮT (Chỉ xem mẫu)'}")
if PROCEED_UPSERT:
    print(json.dumps(PIPELINE_STATS, ensure_ascii=False, indent=2))


🚀 Đang xử lý tài liệu 1 đến 200 / 5000...


Inference Embeddings: 100%|██████████| 30/30 [01:36<00:00,  3.21s/it]


✅ Xong batch! Lũy kế: 3,782 chunks.

🚀 Đang xử lý tài liệu 201 đến 400 / 5000...


Inference Embeddings: 100%|██████████| 43/43 [02:34<00:00,  3.60s/it]


✅ Xong batch! Lũy kế: 9,255 chunks.

🚀 Đang xử lý tài liệu 401 đến 600 / 5000...


...

Inference Embeddings: 100%|██████████| 32/32 [02:13<00:00,  4.17s/it]


✅ Xong batch! Lũy kế: 30,831 chunks.

🚀 Đang xử lý tài liệu 1401 đến 1600 / 5000...


Inference Embeddings: 100%|██████████| 36/36 [02:07<00:00,  3.55s/it]


✅ Xong batch! Lũy kế: 35,352 chunks.

🚀 Đang xử lý tài liệu 1601 đến 1800 / 5000...


Inference Embeddings:  81%|████████  | 21/26 [01:35<00:10,  2.03s/it]

In [ ]:
import time

def analyze_appendix_ratio(dataset, metadata_dict):
    print("🔍 Đang quét dữ liệu để phân tích tỷ lệ Phụ lục / Nội dung chính...")
    t_start = time.perf_counter()

    total_docs = len(dataset)
    total_chunks = 0
    appendix_chunks = 0
    main_chunks = 0
    doc_count = 0

    for row in dataset:
        doc_id = str(row["id"])
        meta = metadata_dict.get(doc_id)
        if not meta:
            continue

        text_content = str(row.get("text", row.get("content", "")))

        # Gọi chunker để cắt thử
        doc_chunks = chunker.process_document(text_content, meta)

        for chunk in doc_chunks:
            total_chunks += 1
            # Kiểm tra cờ is_appendix
            if chunk["metadata"].get("is_appendix") == True:
                appendix_chunks += 1
            else:
                main_chunks += 1

        doc_count += 1
        if doc_count % 500 == 0:
            print(f"--- Đã quét {doc_count}/{total_docs} văn bản... ---")

    duration = time.perf_counter() - t_start

    # Tính phần trăm
    app_percent = (appendix_chunks / total_chunks) * 100 if total_chunks > 0 else 0
    main_percent = (main_chunks / total_chunks) * 100 if total_chunks > 0 else 0

    print("\n" + "="*60)
    print("📊 BÁO CÁO TỶ LỆ PHỤ LỤC (APPENDIX RATIO REPORT)")
    print("="*60)
    print(f"🔹 Tổng số văn bản đã quét: {doc_count:,}")
    print(f"🔹 Tổng số chunks sinh ra:  {total_chunks:,}")
    print("-" * 60)
    print(f"🟢 NỘI DUNG CHÍNH (is_appendix=False): {main_chunks:>7,} chunks ({main_percent:>5.2f}%)")
    print(f"🟡 PHỤ LỤC (is_appendix=True):        {appendix_chunks:>7,} chunks ({app_percent:>5.2f}%)")
    print("-" * 60)
    print(f"⏱️ Thời gian tính toán: {duration:.2f} giây")
    print("="*60)

    # Đưa ra chẩn đoán dựa trên tỷ lệ
    if app_percent > 35:
        print("⚠️ CẢNH BÁO ĐỎ: DB của bạn đang bị 'ngập' phụ lục!")
        print("💡 Nguy cơ: Tốn dung lượng Qdrant, làm loãng kết quả tìm kiếm ngữ nghĩa.")
    elif app_percent > 15:
        print("⚡ LƯU Ý: Tỷ lệ phụ lục ở mức trung bình cao. Cần dùng Filter khi truy vấn.")
    else:
        print("✅ TUYỆT VỜI: Tỷ lệ nội dung chính áp đảo, DB rất 'sạch' và chất lượng!")

# Chạy thử nghiệm
analyze_appendix_ratio(ds_content_selected, metadata_by_id)